In [11]:
# %pip install optuna

In [12]:
import warnings
from functools import partial

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFECV
from sklearn.experimental import enable_halving_search_cv
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import MinMaxScaler, FunctionTransformer
from sklearn.base import TransformerMixin, RegressorMixin, BaseEstimator
from sklearn.model_selection import TimeSeriesSplit, HalvingRandomSearchCV, train_test_split, GridSearchCV, ParameterGrid
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error, make_scorer
from xgboost import XGBRegressor
import inspect

import lightgbm as lgb
import optuna

import requests
from functools import reduce

from scipy.stats import loguniform, randint, uniform

from google.colab import drive
drive.mount('/content/drive')

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

[I 2025-11-07 10:03:05,163] Trial 2 pruned. 


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
def MirrorLog(X, c:float=(1/3)):

    '''
    X   :   The data that will be mirror-log transformed.
    c   :   Constant parameter. Default is 1/3, as suggested in the literature.
    '''

    X = np.asarray(X)
    X_new = np.empty_like(X)
    X_new[X != 0] = np.sign(X[X != 0]) * (np.log(np.abs(X[X != 0]) * (1/c)) + np.log(c))
    X_new[X == 0] = 0
    return X_new

# reverses the mirror log transformation, restoring data—including negative or zero values—back to its original scale by undoing the signed logarithmic mapping.
def InverseMirrorLog(X, c:float=(1/3)):

    '''
    X   :   The data that will be inverse mirror-log transformed.
    c   :   Constant parameter. Default is 1/3, as suggested in the literature.
    '''

    X = np.asarray(X)
    X_inv = np.empty_like(X)
    X_inv[X != 0] = np.sign(X[X != 0]) * (np.exp(np.abs(X[X != 0]) - np.log(c)) - (1/c))
    X_inv[X == 0] = 0
    return X_inv


class MirrorLogNormScaler(TransformerMixin, BaseEstimator):

    def __init__(self, mirrorlog_kwargs:dict=None, normalizer=None):

        '''
        mirrorlog_kwargs    :   Keyword arguments for the mirror-log transformation. Default is {'c': 1/3}.
        normalizer          :   Normalizer to use after mirror-log transformation. Default is MinMaxScaler.
        '''

        self.mirrorlog_kwargs = mirrorlog_kwargs
        self.normalizer = normalizer
        _normalizer = self.normalizer or MinMaxScaler()
        self._normalizer = _normalizer

        mirrorlog_kwargs_ = self.mirrorlog_kwargs or {'c': 1/3}

        self._mlog_scaler = FunctionTransformer(
            func=partial(MirrorLog, **mirrorlog_kwargs_),
            inverse_func=partial(InverseMirrorLog, **mirrorlog_kwargs_),
            check_inverse=False
        )

    def fit(self, X, y=None):

        '''
            X   :   The data that will be mirror-log transformed then used to compute the per-feature minimum and maximum used for later scaling along the features axis.
            y   :   Ignored.
        '''

        X_mlog = self._mlog_scaler.fit_transform(X)
        self._normalizer.fit(X_mlog)
        return self

    def transform(self, X):

        '''
        X   :   The data that will be mirror-log transformed then min-max scaled.
        '''

        X_mlog = self._mlog_scaler.transform(X)
        X_new = self._normalizer.transform(X_mlog)
        return X_new

    def inverse_transform(self, X):

        '''
        X   :   The data that will be inverse min-max scaled then inverse mirror-log transformed.
        '''

        X_mlog = self._normalizer.inverse_transform(X)
        X_inv = self._mlog_scaler.inverse_transform(X_mlog)
        return X_inv


class ProcessingPipeline(RegressorMixin, BaseEstimator):

    '''
    rfecv_kwargs        :   Keyword arguments for the RFECV feature selection step. Estimator for feature importance must be provided. Default is a LGBM regressor with 3-fold CV and step size of 2.
    scaler              :   Scaler for feature normalization. Default is MinMaxScaler.
    estimator           :   Estimator for the regression task. Default is a LGBM regressor.
    target_transformer  :   Transformer (or scaler) for the target variable. If set to 'ignore', no transformation is applied. Default is MirrorLogNormScaler. Transformer must implement fit, transform, and inverse_transform methods.
    '''

    def __init__(self, rfecv_kwargs:dict=None, scaler=None, estimator=None, target_transformer=None):

        self.rfecv_kwargs = rfecv_kwargs
        self.scaler = scaler
        self.estimator = estimator
        self.target_transformer = target_transformer

        rfecv_kwargs = self.rfecv_kwargs or {'estimator':lgb.LGBMRegressor(verbose=-1), 'cv':3, 'step':2}
        scaler_ = self.scaler or MinMaxScaler()
        estimator_ = self.estimator or lgb.LGBMRegressor(verbose=-1)
        target_transformer_ = self.target_transformer or MirrorLogNormScaler()

        if self.rfecv_kwargs in (None, False, 'ignore', 'skip'):
            feature_step = ('feature_elimination', 'passthrough')
        else:
            feature_step = ('feature_elimination', RFECV(**self.rfecv_kwargs))

        pipe = Pipeline([
            ('feature_elimination', RFECV(**rfecv_kwargs)),
            ('normalization', scaler_),
            ('estimation', estimator_)
        ])

        if self.target_transformer != 'ignore':
            ttr_pipe = TransformedTargetRegressor(
                regressor=pipe,
                transformer=target_transformer_,
                check_inverse=False
            )
        else:
            ttr_pipe = None
        self.model = ttr_pipe if ttr_pipe else pipe

    def fit(self, X, y):

        '''
        X   :   Training matrix.
        y   :   Target values.
        '''

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            self.model.fit(X, y)
        return self

    def predict(self, X):

        '''
        X   :   Samples.
        '''

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            return self.model.predict(X)

def willmotts_index(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    wi = 1 - (np.sum((y_true-y_pred)**2) / np.sum((np.abs(y_pred-np.mean(y_true))+(np.abs(y_true-np.mean(y_pred))))**2))
    return wi

def nash_sutcliffe_efficiency(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ns = 1 - np.sum((y_true-y_pred)**2) / np.sum((y_true-np.mean(y_true))**2)
    return ns

def legates_mccabes_index(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    lm = 1 - np.sum(np.abs(y_pred-y_true)) / np.sum(np.abs(y_true-np.mean(y_true)))
    return lm

def kling_gupta_efficiency(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    cv_true = np.std(y_true) / np.mean(y_true)
    cv_pred = np.std(y_pred) / np.mean(y_pred)
    r = np.sum((y_true - y_true.mean()) * (y_pred - y_pred.mean())) / np.sqrt(np.sum((y_true - y_true.mean())**2) * np.sum((y_pred - y_pred.mean())**2))
    kge = 1 - np.sqrt((r-1)**2 + (np.mean(y_pred)/np.mean(y_true) - 1)**2 + (cv_pred/cv_true)**2)
    return kge

def normalized_root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    nrmse = root_mean_squared_error(y_true=y_true, y_pred=y_pred) / np.mean(y_true)
    return nrmse

def relative_mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rmae = mean_absolute_error(y_true=y_true, y_pred=y_pred) / np.mean(y_true)
    return rmae

def symmetric_mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    smape = (1/len(y_true)) * np.sum(np.abs(y_true-y_pred) / ((np.abs(y_true) + np.abs(y_pred))/2))
    return smape

def theils_inequality_coefficient(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    numerator = np.sqrt((1/n) * (np.sum(y_pred-y_true)**2))
    denominator = np.sqrt((1/n) * np.sum(y_true**2)) + np.sqrt((1/n) * np.sum(y_pred**2))
    tic = numerator / denominator
    return tic

def absolute_percentage_bias(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    apb = np.abs(np.sum(y_true-y_pred) / np.sum(y_true))
    return apb

def evaluate_model(model, X, y_true) -> pd.DataFrame:
    '''
    model   :   A pre-trained model object.
    X       :   Feature matrix. Should be in the format your model object requires.
    y_true  :   True target array for prediction evaluation.
    '''

    if hasattr(model, 'predict'):
        y_pred = model.predict(X)
    elif callable(model):
        y_pred = model(X)
    else:
        raise TypeError('Model must be callable or have a .predict() method.')

    abbr = ['rmse', 'nrmse']
    prop = ['bias', 'bias']
    metrics = [root_mean_squared_error, normalized_root_mean_squared_error]

    results = [metric(y_true=y_true, y_pred=y_pred) for metric in metrics]

    df = pd.DataFrame(
        data=list(zip(prop, abbr, results)),
        columns=['property', 'metric', 'score']
    )

    return df

In [14]:
# Replace 'your_file_path.csv' with the actual path to your CSV file in Google Drive
csv_file_path = '/content/drive/MyDrive/MURES/rfe_dataset_2019_2025.csv'
df_rfe = pd.read_csv(csv_file_path)

# Setting Index
df_rfe['datetime'] = pd.to_datetime(df_rfe['datetime'], utc=True)
df_rfe = df_rfe.set_index('datetime')

y = df_rfe['price']
X_lag = df_rfe.drop(columns=['price'])

nrmse_scorer = make_scorer(normalized_root_mean_squared_error, greater_is_better=False)


[I 2025-11-07 10:03:07,291] Trial 11 pruned. 


In [15]:
# Removing NA values
df_xy = X_lag.copy()
df_xy["__y__"] = y
df_xy = df_xy.dropna()

y_clean = df_xy["__y__"]
X_clean = df_xy.drop(columns="__y__")

# Split the Data into Train and Test
X_train_val, X_test, y_train_val, y_test = train_test_split(X_clean, y_clean, test_size=0.1, shuffle=False)

train_size = int(len(X_train_val) * (0.9*0.8))  # 0.9 because train needs to be 80% of train/val dataset & validation shoudld be 90% of total dataset
step_size = 24*7*20  # every fold is 20 weeks in length
n_splits = (len(X_train_val) - train_size) // step_size
tscv = TimeSeriesSplit(n_splits=n_splits, max_train_size=train_size)

def _fit_supports(argname, estimator):
    """Check if the estimator.fit() method supports a given argument name."""
    sig = inspect.signature(estimator.fit)
    return argname in sig.parameters

In [18]:
## Hyperparameter tuning optimization using Optuna

def objective(trial):

    # XGBoost parameter
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_categorical("reg_alpha", [0.0, 1e-8, 1e-6, 1e-4, 1e-3]),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
    }

    # 각 trial에서 교차검증 + 조기종료 반영 성능 계산
    fold_scores = []
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train_val, y_train_val)):
        Xtr, Xva = X_train_val.iloc[tr_idx], X_train_val.iloc[va_idx]
        ytr, yva = y_train_val.iloc[tr_idx], y_train_val.iloc[va_idx]

        model = XGBRegressor(
            n_estimators=300,
            tree_method='hist',
            random_state=42,
            n_jobs= 1,
            objective='reg:squarederror',
            eval_metric='rmse',
            **params
        )

        fit_kwargs = dict(
            X=Xtr, y=ytr,
            eval_set=[(Xva, yva)],
            verbose=False
        )

        # EarlyStopping: callbacks 우선, 없으면 early_stopping_rounds
        if _fit_supports("callbacks", model):
            from xgboost.callback import EarlyStopping
            fit_kwargs["callbacks"] = [EarlyStopping(rounds=100, save_best=True, maximize=False)]
        elif _fit_supports("early_stopping_rounds", model):
            fit_kwargs["early_stopping_rounds"] = 100

        model.fit(**fit_kwargs)

        # 사용자가 넘긴 nrmse_scorer(호출 가능 객체)로 fold 성능 계산
        score = nrmse_scorer(model, Xva, yva) if callable(getattr(nrmse_scorer, '__call__', None)) else float('nan')
        fold_scores.append(score)

        # Optuna에 중간 리포트 & Pruning 기회 제공
        trial.report(float(np.nanmean(fold_scores)), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    # 이 프로젝트의 스코어는 "클수록 좋음" 형태이므로 그대로 반환
    return float(np.nanmean(fold_scores))

# Study 생성 (TPE + Hyperband Pruner)
study = optuna.create_study(
    direction='maximize',  # 점수가 클수록 좋음
    sampler=optuna.samplers.TPESampler(seed=391),
    pruner=optuna.pruners.HyperbandPruner()
    # 필요 시 storage=... , study_name=... , load_if_exists=True 추가 가능
)

# 최적화 실행 (병렬/시간 제한 원하는 대로 조정)
study.optimize(objective, n_trials=256, n_jobs=16)  # timeout=60*60*24 등도 가능

print("[Optuna] Best score:", study.best_value)
print("[Optuna] Best params:", study.best_params)

best_params = study.best_params
best_score = study.best_value

# =========================
# ✅ 최종 학습 (Best params로 전체 train_val 사용, 내부 10%를 검증으로 조기종료)
# =========================
final_model = XGBRegressor(
    n_estimators=1000,
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
    objective='reg:squarederror',
    eval_metric='rmse',
    **best_params
)

split = int(len(X_train_val) * 0.9)
fit_kwargs = dict(
    X=X_train_val.iloc[:split], y=y_train_val.iloc[:split],
    eval_set=[(X_train_val.iloc[split:], y_train_val.iloc[split:])],
    verbose=False
)
if _fit_supports("callbacks", final_model):
    from xgboost.callback import EarlyStopping
    fit_kwargs["callbacks"] = [EarlyStopping(rounds=100, save_best=True, maximize=False)]
elif _fit_supports("early_stopping_rounds", final_model):
    fit_kwargs["early_stopping_rounds"] = 100

final_model.fit(**fit_kwargs)

class SearchLike:
    best_estimator_ = final_model
    best_params_    = best_params
    best_score_     = best_score
    study_          = study

search_es = SearchLike()
print("BEST:", search_es.best_score_, search_es.best_params_)



[I 2025-11-07 10:38:23,645] A new study created in memory with name: no-name-ebd37aa6-4333-416e-8aba-78bb326a9fc3
[I 2025-11-07 10:39:31,819] Trial 5 finished with value: -0.6143607583998346 and parameters: {'max_depth': 3, 'learning_rate': 0.19914090473259655, 'subsample': 0.7430362910314112, 'colsample_bytree': 0.6194180090591476, 'min_child_weight': 2, 'reg_alpha': 0.0001, 'reg_lambda': 0.018607322370076812}. Best is trial 5 with value: -0.6143607583998346.
[I 2025-11-07 10:39:35,200] Trial 15 finished with value: -0.5506063640459834 and parameters: {'max_depth': 3, 'learning_rate': 0.017450984269200977, 'subsample': 0.9727667065858777, 'colsample_bytree': 0.7354804445631494, 'min_child_weight': 4, 'reg_alpha': 1e-06, 'reg_lambda': 0.06012575112429979}. Best is trial 15 with value: -0.5506063640459834.
[I 2025-11-07 10:40:10,246] Trial 8 finished with value: -0.581694923717928 and parameters: {'max_depth': 5, 'learning_rate': 0.08807956604681487, 'subsample': 0.6587976205527759, 'co

[Optuna] Best score: -0.5375638602673599
[Optuna] Best params: {'max_depth': 3, 'learning_rate': 0.04290438769421161, 'subsample': 0.799890423723672, 'colsample_bytree': 0.77209708002315, 'min_child_weight': 10, 'reg_alpha': 0.0, 'reg_lambda': 0.03204323977238229}
BEST: -0.5375638602673599 {'max_depth': 3, 'learning_rate': 0.04290438769421161, 'subsample': 0.799890423723672, 'colsample_bytree': 0.77209708002315, 'min_child_weight': 10, 'reg_alpha': 0.0, 'reg_lambda': 0.03204323977238229}


In [19]:
evaluate_model(final_model, X_test, y_test)

,property,metric,score
0,bias,rmse,45.291657
1,bias,nrmse,0.519783
